## __Imports and Loads__

In [46]:
# imports
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split#, time_series_split
from lazypredict import LazyRegressor
import xgboost
import lightgbm
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, BaggingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import shap

import requests
import time


In [47]:
pd.set_option('display.max_columns', None)

In [48]:
# load data
df = pd.read_parquet('../data/processed/02_df_cleaned.parquet')

In [49]:
df.head()

,store_id,item_id,date,category_name,sold_quantity,revenue,zipcode,holiday_name,holiday_type,wind_dir,weather_code,temperature,wind_speed,wind_degree,precip,humidity,visibility,pressure,cloudcover,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,rolling_sum_7,rolling_sum_28,expanding_mean,price_rounded
0,0,5,2025-02-01,Kuchen,1.0,10.50,52062,no_holiday,no_holiday,SE,113,1.7,7.300000,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.314970,1.0,3.0,0.001224,10.5
1,0,5,2025-03-15,Kuchen,1.0,10.50,52062,no_holiday,no_holiday,NE,116,2.5,15.400000,42.500000,0.0,71.000000,9.0,1017.500000,49.599998,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.001604,10.5
2,0,5,2025-03-29,Kuchen,1.0,10.50,52062,no_holiday,no_holiday,NNW,176,7.4,14.200000,324.500000,0.1,70.599998,9.0,1018.000000,53.500000,0.0,0.0,1.0,0.0,0.000000,0.000000,0.071429,0.267261,0.035714,0.188982,0.0,1.0,0.001994,10.5
3,0,5,2025-05-17,Kuchen,1.0,11.25,52062,no_holiday,no_holiday,N,113,11.5,8.500000,304.399994,0.0,71.300003,8.0,1019.400024,24.799999,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.002347,11.2
4,0,5,2025-05-24,Kuchen,1.0,11.25,52062,no_holiday,no_holiday,SSW,116,10.1,19.700001,214.199997,0.2,74.800003,8.9,1016.500000,78.099998,0.0,1.0,0.0,0.0,0.142857,0.377964,0.071429,0.267261,0.035714,0.188982,1.0,1.0,0.002731,11.2


## __External Features__

### __New Feature 01: Local events__

In [50]:
# load events
events = pd.read_parquet('../data/processed/events_transformed.parquet', engine='pyarrow')


In [51]:
# check dtype
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     49 non-null     datetime64[us]
 1   event    49 non-null     category      
 2   zipcode  49 non-null     category      
dtypes: category(2), datetime64[us](1)
memory usage: 820.0 bytes


In [52]:
events.head()

,date,event,zipcode
0,2025-09-18,Stadtfest,52062
1,2025-09-19,Stadtfest,52062
2,2025-09-20,Stadtfest,52062
3,2025-09-21,Stadtfest,52062
4,2025-09-22,Stadtfest,52062


In [53]:
# add variable events to original data
df_events = df.merge(events, how='left', on=['date', 'zipcode'])

In [54]:
# replace missing values
df_events['event'] = df_events.event.astype(str).fillna('no_event')

In [55]:
# removing variables unsuited for training
### revenue -> calculated from price and target sold_quantity, thus contains direct information of target
### date -> is already split into calender features

df_events = df_events.drop(columns=['date', 'revenue'])

In [56]:
# change dtypes for modeling
cols_to_change = df_events.select_dtypes(['int64', 'str']).columns

df_events[cols_to_change] = df_events[cols_to_change].astype('category')
df_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 3449803 entries, 0 to 3449802
Data columns (total 32 columns):
 #   Column           Dtype   
---  ------           -----   
 0   store_id         category
 1   item_id          category
 2   category_name    category
 3   sold_quantity    float32 
 4   zipcode          category
 5   holiday_name     category
 6   holiday_type     category
 7   wind_dir         category
 8   weather_code     category
 9   temperature      float32 
 10  wind_speed       float32 
 11  wind_degree      float32 
 12  precip           float32 
 13  humidity         float32 
 14  visibility       float32 
 15  pressure         float32 
 16  cloudcover       float32 
 17  lag_1            float32 
 18  lag_7            float32 
 19  lag_14           float32 
 20  lag_28           float32 
 21  rolling_mean_7   float32 
 22  rolling_std_7    float32 
 23  rolling_mean_14  float32 
 24  rolling_std_14   float32 
 25  rolling_mean_28  float32 
 26  rolling_std_28   float32 

### __New Feature 02: population_count__

In [57]:
# load population_2025
population_count = pd.read_parquet('../data/processed/population_2025_transformed.parquet', engine='pyarrow')
population_count.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   zipcode       10 non-null     str    
 1   Pers_ insges  10 non-null     float64
 2   Deutsch       10 non-null     float64
 3   Ausl          10 non-null     float64
dtypes: float64(3), str(1)
memory usage: 502.0 bytes


In [58]:
population_count[['Pers_ insges', 'Deutsch', 'Ausl']] = population_count[['Pers_ insges', 'Deutsch', 'Ausl']].astype(int)
population_count.describe()

,Pers_ insges,Deutsch,Ausl
count,10.000000,10.000000,10.000000
mean,26276.500000,19750.600000,6525.900000
std,8757.381423,7344.784154,3075.784758
min,11292.000000,5721.000000,1420.000000
25%,21050.000000,17040.500000,5149.500000
50%,27533.000000,19009.000000,6512.000000
75%,30494.250000,22884.750000,8274.250000
max,39815.000000,31286.000000,10795.000000


In [59]:
population_count.head()

,zipcode,Pers_ insges,Deutsch,Ausl
0,52062,28632,17837,10795
1,52064,26434,18924,7510
2,52066,25742,20733,5009
3,52068,11292,5721,5571
4,52070,29784,19094,10690


In [60]:
# merge with origin data

df_population_count = df.merge(population_count, how='left', on='zipcode')

In [61]:
df_population_count = df_population_count.drop(columns=['date', 'revenue'])

In [62]:
df_population_count.info()

<class 'pandas.DataFrame'>
RangeIndex: 3449803 entries, 0 to 3449802
Data columns (total 34 columns):
 #   Column           Dtype   
---  ------           -----   
 0   store_id         int64   
 1   item_id          int64   
 2   category_name    category
 3   sold_quantity    float32 
 4   zipcode          str     
 5   holiday_name     category
 6   holiday_type     category
 7   wind_dir         category
 8   weather_code     int64   
 9   temperature      float32 
 10  wind_speed       float32 
 11  wind_degree      float32 
 12  precip           float32 
 13  humidity         float32 
 14  visibility       float32 
 15  pressure         float32 
 16  cloudcover       float32 
 17  lag_1            float32 
 18  lag_7            float32 
 19  lag_14           float32 
 20  lag_28           float32 
 21  rolling_mean_7   float32 
 22  rolling_std_7    float32 
 23  rolling_mean_14  float32 
 24  rolling_std_14   float32 
 25  rolling_mean_28  float32 
 26  rolling_std_28   float32 

In [63]:
# dtypes
cols_to_change = df_population_count.select_dtypes(['str', 'int64']).columns
df_population_count[cols_to_change] = df_population_count[cols_to_change].astype('category')
df_population_count
#--------------------------------
cols_to_change = df_population_count.select_dtypes(['float64']).columns
df_population_count[cols_to_change] = df_population_count[cols_to_change].astype('float32')

In [64]:
df_population_count.info()
df_population_count.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3449803 entries, 0 to 3449802
Data columns (total 34 columns):
 #   Column           Dtype   
---  ------           -----   
 0   store_id         category
 1   item_id          category
 2   category_name    category
 3   sold_quantity    float32 
 4   zipcode          category
 5   holiday_name     category
 6   holiday_type     category
 7   wind_dir         category
 8   weather_code     category
 9   temperature      float32 
 10  wind_speed       float32 
 11  wind_degree      float32 
 12  precip           float32 
 13  humidity         float32 
 14  visibility       float32 
 15  pressure         float32 
 16  cloudcover       float32 
 17  lag_1            float32 
 18  lag_7            float32 
 19  lag_14           float32 
 20  lag_28           float32 
 21  rolling_mean_7   float32 
 22  rolling_std_7    float32 
 23  rolling_mean_14  float32 
 24  rolling_std_14   float32 
 25  rolling_mean_28  float32 
 26  rolling_std_28   float32 

store_id                 0
item_id                  0
category_name            0
sold_quantity            0
zipcode                  0
holiday_name             0
holiday_type             0
wind_dir                 0
weather_code             0
temperature              0
wind_speed               0
wind_degree              0
precip                   0
humidity                 0
visibility               0
pressure                 0
cloudcover               0
lag_1                    0
lag_7                    0
lag_14                   0
lag_28                   0
rolling_mean_7           0
rolling_std_7            0
rolling_mean_14          0
rolling_std_14           0
rolling_mean_28          0
rolling_std_28           0
rolling_sum_7            0
rolling_sum_28           0
expanding_mean           0
price_rounded            0
Pers_ insges       1906864
Deutsch            1906864
Ausl               1906864
dtype: int64

### __New Feature 03: Age Groups__

In [65]:
age_groups = pd.read_parquet('../data/processed/age_groups_2025_transformed.parquet', engine='pyarrow').reset_index()
age_groups


,zipcode,0-2,3-5,6-9,10-14,15-17,18-19,20-24,25-29,30-34,35-39,40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90um
0,52062,430.0,318.0,369.0,421.0,242.0,999.0,6744.0,6312.0,3521.0,1846.0,1156.0,929.0,775.0,950.0,920.0,762.0,604.0,544.0,355.0,308.0,127.0
1,52064,413.0,331.0,428.0,499.0,306.0,763.0,5022.0,5633.0,3313.0,1669.0,1169.0,925.0,864.0,1058.0,1016.0,892.0,660.0,474.0,392.0,398.0,209.0
2,52066,586.0,569.0,677.0,787.0,536.0,552.0,2686.0,3221.0,2659.0,1877.0,1404.0,1260.0,1254.0,1581.0,1689.0,1327.0,979.0,746.0,599.0,541.0,212.0
3,52068,279.0,310.0,416.0,476.0,287.0,286.0,1251.0,1291.0,1028.0,764.0,704.0,689.0,606.0,656.0,633.0,519.0,352.0,291.0,229.0,154.0,71.0
4,52070,591.0,582.0,835.0,1057.0,602.0,826.0,4145.0,4815.0,3054.0,2092.0,1699.0,1430.0,1333.0,1515.0,1354.0,1111.0,837.0,737.0,513.0,474.0,182.0
5,52072,416.0,516.0,724.0,834.0,508.0,385.0,934.0,1078.0,1198.0,1320.0,1212.0,1120.0,1038.0,1498.0,1532.0,1360.0,1126.0,927.0,788.0,645.0,327.0
6,52074,598.0,701.0,1015.0,1194.0,787.0,819.0,3526.0,3234.0,2278.0,1781.0,1610.0,1368.0,1349.0,1842.0,1933.0,1765.0,1479.0,1180.0,960.0,874.0,438.0
7,52076,381.0,498.0,586.0,703.0,452.0,281.0,639.0,592.0,744.0,1000.0,907.0,863.0,845.0,1349.0,1350.0,1142.0,964.0,785.0,651.0,541.0,231.0
8,52078,936.0,1136.0,1664.0,2073.0,1236.0,832.0,2199.0,2557.0,2554.0,2520.0,2465.0,2398.0,2457.0,2813.0,3000.0,2566.0,2096.0,1628.0,1254.0,1005.0,426.0
9,52080,763.0,952.0,1365.0,1626.0,1010.0,733.0,2029.0,2212.0,2228.0,2205.0,2206.0,2002.0,2021.0,2774.0,2765.0,2275.0,1894.0,1511.0,1249.0,1046.0,479.0


In [66]:
# merge with origin

df_age_groups = df.merge(age_groups, on='zipcode', how='left').drop(columns=['date', 'revenue'])

In [67]:
# zipcoed with missing age groups information
missing_ag = df_age_groups.isna().sum()
print(missing_ag[missing_ag > 0])
df_age_groups[df_age_groups.isna().any(axis=1)].zipcode.unique()


0-2      1906864
3-5      1906864
6-9      1906864
10-14    1906864
15-17    1906864
18-19    1906864
20-24    1906864
25-29    1906864
30-34    1906864
35-39    1906864
40-44    1906864
45-49    1906864
50-54    1906864
55-59    1906864
60-64    1906864
65-69    1906864
70-74    1906864
75-79    1906864
80-84    1906864
85-89    1906864
90um     1906864
dtype: int64


<ArrowStringArray>
['52224', '52134', '52222', '52379', '52146', '52249', '52477', '52223',
 '52499', '52428', '52152', '52531', '52159', '52156', '52382', '52351']
Length: 16, dtype: str

In [68]:
# check dtypes
cols_to_change = df_age_groups.select_dtypes('int', 'str').columns
df_age_groups[cols_to_change] = df_age_groups[cols_to_change].astype('category')
df_age_groups['zipcode'] = df_age_groups['zipcode'].astype('category')
df_age_groups.dtypes

store_id           category
item_id            category
category_name      category
sold_quantity       float32
zipcode            category
holiday_name       category
holiday_type       category
wind_dir           category
weather_code       category
temperature         float32
wind_speed          float32
wind_degree         float32
precip              float32
humidity            float32
visibility          float32
pressure            float32
cloudcover          float32
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
rolling_mean_7      float32
rolling_std_7       float32
rolling_mean_14     float32
rolling_std_14      float32
rolling_mean_28     float32
rolling_std_28      float32
rolling_sum_7       float32
rolling_sum_28      float32
expanding_mean      float32
price_rounded       float32
0-2                 float64
3-5                 float64
6-9                 float64
10-14               float64
15-17               

### __New Features 04: Marital status__

In [69]:
# load
marital_status = pd.read_parquet('../data/processed/marital_status_transformed.parquet', engine='pyarrow').reset_index()
marital_status.head(2)

,zipcode,LD,VH,VW,GS
0,52062,21824.0,5015.0,598.0,1195.0
1,52064,19299.0,5120.0,778.0,1237.0


In [70]:
# merge with origin

df_marital_status = df.merge(marital_status, on='zipcode', how='left').drop(columns=['date', 'revenue'])
df_marital_status.head(2)

,store_id,item_id,category_name,sold_quantity,zipcode,holiday_name,holiday_type,wind_dir,weather_code,temperature,wind_speed,wind_degree,precip,humidity,visibility,pressure,cloudcover,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,rolling_sum_7,rolling_sum_28,expanding_mean,price_rounded,LD,VH,VW,GS
0,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,21824.0,5015.0,598.0,1195.0
1,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,NE,116,2.5,15.4,42.500000,0.0,71.000000,9.0,1017.500000,49.599998,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.001604,10.5,21824.0,5015.0,598.0,1195.0


In [71]:
missing_ma = df_marital_status.isnull().sum()
missing_ma[missing_ma > 0]

LD    1906864
VH    1906864
VW    1906864
GS    1906864
dtype: int64

In [72]:
# dtypes
# check dtypes
cols_to_change = df_marital_status.select_dtypes('int', 'str').columns
df_marital_status[cols_to_change] = df_marital_status[cols_to_change].astype('category')
df_marital_status['zipcode'] = df_marital_status['zipcode'].astype('category')
df_marital_status.dtypes

store_id           category
item_id            category
category_name      category
sold_quantity       float32
zipcode            category
holiday_name       category
holiday_type       category
wind_dir           category
weather_code       category
temperature         float32
wind_speed          float32
wind_degree         float32
precip              float32
humidity            float32
visibility          float32
pressure            float32
cloudcover          float32
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
rolling_mean_7      float32
rolling_std_7       float32
rolling_mean_14     float32
rolling_std_14      float32
rolling_mean_28     float32
rolling_std_28      float32
rolling_sum_7       float32
rolling_sum_28      float32
expanding_mean      float32
price_rounded       float32
LD                  float64
VH                  float64
VW                  float64
GS                  float64
dtype: object

### __New Features 05: Employed (2024!)__

In [73]:
employed_2024 = pd.read_parquet('../data/processed/employed_2024_transformed.parquet', engine='pyarrow')

In [74]:
print(employed_2024.columns)
employed_2024.head()


Index(['dist_id', 'dist_name', 'total', 'male', 'female', 'german',
       'non-german', '<25', '25-35', '>35',
       'Total_share_working_age_population', 'zipcode'],
      dtype='str')


,dist_id,dist_name,total,male,female,german,non-german,<25,25-35,>35,Total_share_working_age_population,zipcode
0,10,Markt,1.281,748,533,886,395,233,622,426,"52,69%",52062
1,13,Theater,1.333,790,543,939,394,192,678,463,"48,86%",52062
2,14,Lindenplatz,1.693,1.024,669,1.200,493,273,858,562,"49,04%",52064
3,15,St. Jakob,2.967,1.680,1.287,2.328,639,471,1.515,981,"52,70%",52064
4,16,Westpark,3.520,2.013,1.507,2.649,871,471,1.591,1.458,"52,96%",52064


In [75]:
# 1 total employed
df_employed_total = df.merge(employed_2024[['zipcode', 'total']], on='zipcode', how='left').drop(columns=['date', 'revenue'])

df_employed_total = df_employed_total.rename(columns={'total': 'total_employed'})
display(df_employed_total.head())

,store_id,item_id,category_name,sold_quantity,zipcode,holiday_name,holiday_type,wind_dir,weather_code,temperature,wind_speed,wind_degree,precip,humidity,visibility,pressure,cloudcover,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,rolling_sum_7,rolling_sum_28,expanding_mean,price_rounded,total_employed
0,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,1.281
1,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,1.333
2,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,5.432
3,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.900000,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,3.776
4,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,NE,116,2.5,15.4,42.500000,0.0,71.000000,9.0,1017.500000,49.599998,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.001604,10.5,1.281


In [76]:
# dtypes
# check dtypes employed_total
cols_to_change = df_employed_total.select_dtypes('int', 'str').columns
df_employed_total[cols_to_change] = df_employed_total[cols_to_change].astype('category')
df_employed_total[['zipcode', 'total_employed']] = df_employed_total[['zipcode', 'total_employed']].astype('category')
df_employed_total.dtypes

store_id           category
item_id            category
category_name      category
sold_quantity       float32
zipcode            category
holiday_name       category
holiday_type       category
wind_dir           category
weather_code       category
temperature         float32
wind_speed          float32
wind_degree         float32
precip              float32
humidity            float32
visibility          float32
pressure            float32
cloudcover          float32
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
rolling_mean_7      float32
rolling_std_7       float32
rolling_mean_14     float32
rolling_std_14      float32
rolling_mean_28     float32
rolling_std_28      float32
rolling_sum_7       float32
rolling_sum_28      float32
expanding_mean      float32
price_rounded       float32
total_employed     category
dtype: object

In [77]:
# 2 age employed
df_employed_age = df.merge(employed_2024[['zipcode', '<25', '25-35', '>35']], on='zipcode', how='left').drop(columns=['date', 'revenue'])
df_employed_age = df_employed_age.rename(columns={'<25':'emp_under_25', '25-35': 'emp_25-35', '>35': 'emp_above_35'})

display(df_employed_age.head(2))

,store_id,item_id,category_name,sold_quantity,zipcode,holiday_name,holiday_type,wind_dir,weather_code,temperature,wind_speed,wind_degree,precip,humidity,visibility,pressure,cloudcover,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,rolling_mean_28,rolling_std_28,rolling_sum_7,rolling_sum_28,expanding_mean,price_rounded,emp_under_25,emp_25-35,emp_above_35
0,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.9,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,233,622,426
1,0,5,Kuchen,1.0,52062,no_holiday,no_holiday,SE,113,1.7,7.3,115.900002,0.0,75.099998,10.0,1030.099976,2.9,0.0,1.0,1.0,1.0,0.142857,0.377964,0.142857,0.363137,0.107143,0.31497,1.0,3.0,0.001224,10.5,192,678,463


In [78]:
# dtypes
# check dtypes employed age
cols_to_change = df_employed_age.select_dtypes('int', 'str').columns
df_employed_age[cols_to_change] = df_employed_age[cols_to_change].astype('category')
df_employed_age[['zipcode', 'emp_under_25', 'emp_25-35', 'emp_above_35']] = df_employed_age[['zipcode', 'emp_under_25', 'emp_25-35', 'emp_above_35']].astype('category')
df_employed_age.dtypes

store_id           category
item_id            category
category_name      category
sold_quantity       float32
zipcode            category
holiday_name       category
holiday_type       category
wind_dir           category
weather_code       category
temperature         float32
wind_speed          float32
wind_degree         float32
precip              float32
humidity            float32
visibility          float32
pressure            float32
cloudcover          float32
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
rolling_mean_7      float32
rolling_std_7       float32
rolling_mean_14     float32
rolling_std_14      float32
rolling_mean_28     float32
rolling_std_28      float32
rolling_sum_7       float32
rolling_sum_28      float32
expanding_mean      float32
price_rounded       float32
emp_under_25       category
emp_25-35          category
emp_above_35       category
dtype: object

### __New Features 06: Competitors Count by zipcode area__

In [79]:
# ..

### __Check df origin__

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3449803 entries, 0 to 3449802
Data columns (total 33 columns):
 #   Column           Dtype         
---  ------           -----         
 0   store_id         int64         
 1   item_id          int64         
 2   date             datetime64[ns]
 3   category_name    category      
 4   sold_quantity    float32       
 5   revenue          float32       
 6   zipcode          category      
 7   holiday_name     category      
 8   holiday_type     category      
 9   wind_dir         category      
 10  weather_code     int64         
 11  temperature      float32       
 12  wind_speed       float32       
 13  wind_degree      float32       
 14  precip           float32       
 15  humidity         float32       
 16  visibility       float32       
 17  pressure         float32       
 18  cloudcover       float32       
 19  lag_1            float32       
 20  lag_7            float32       
 21  lag_14           float32       
 22  lag_2

In [81]:
df_orig = df.drop(columns=['date', 'revenue'])


In [82]:
cols_to_change = df_orig.select_dtypes(['str', 'int64']).columns
df_orig[cols_to_change] = df_orig[cols_to_change].astype('category')
df_orig.info()

<class 'pandas.DataFrame'>
RangeIndex: 3449803 entries, 0 to 3449802
Data columns (total 31 columns):
 #   Column           Dtype   
---  ------           -----   
 0   store_id         category
 1   item_id          category
 2   category_name    category
 3   sold_quantity    float32 
 4   zipcode          category
 5   holiday_name     category
 6   holiday_type     category
 7   wind_dir         category
 8   weather_code     category
 9   temperature      float32 
 10  wind_speed       float32 
 11  wind_degree      float32 
 12  precip           float32 
 13  humidity         float32 
 14  visibility       float32 
 15  pressure         float32 
 16  cloudcover       float32 
 17  lag_1            float32 
 18  lag_7            float32 
 19  lag_14           float32 
 20  lag_28           float32 
 21  rolling_mean_7   float32 
 22  rolling_std_7    float32 
 23  rolling_mean_14  float32 
 24  rolling_std_14   float32 
 25  rolling_mean_28  float32 
 26  rolling_std_28   float32 

### __Test_dictionary__

In [83]:
# test dict

test_dict = {
    'df_orig': df_orig,
    'df_events': df_events,
    'df_pop_count': df_population_count,
    'df_age_groups': df_age_groups,
    'df_marital_status': df_marital_status,
    'df_employed_total': df_employed_total,
    'df_employed_age': df_employed_age
}




In [84]:
# check check weg
for n, d in test_dict.items():
    print(n, d.isna().sum())

df_orig.shape

df_orig store_id           0
item_id            0
category_name      0
sold_quantity      0
zipcode            0
holiday_name       0
holiday_type       0
wind_dir           0
weather_code       0
temperature        0
wind_speed         0
wind_degree        0
precip             0
humidity           0
visibility         0
pressure           0
cloudcover         0
lag_1              0
lag_7              0
lag_14             0
lag_28             0
rolling_mean_7     0
rolling_std_7      0
rolling_mean_14    0
rolling_std_14     0
rolling_mean_28    0
rolling_std_28     0
rolling_sum_7      0
rolling_sum_28     0
expanding_mean     0
price_rounded      0
dtype: int64
df_events store_id           0
item_id            0
category_name      0
sold_quantity      0
zipcode            0
holiday_name       0
holiday_type       0
wind_dir           0
weather_code       0
temperature        0
wind_speed         0
wind_degree        0
precip             0
humidity           0
visibility         0
pre

(3449803, 31)

### __WMAPE error function__

In [85]:
## Wmape - relative error function

# test data
tr = pd.Series([1, 1, 3, 3, 3, 5, 5, 5])
pr = pd.Series([2, 1, 3, 5, 3, 5, 9, 7])

# funtion
def weighted_mean_absolute_percentage_error(true, pred):
    wmape = np.sum(abs(true - pred)) / np.sum(np.abs(true))
    return wmape

weighted_mean_absolute_percentage_error(tr, pr)

np.float64(0.34615384615384615)

## __Test Loop__

In [86]:
for name, df in test_dict.items():
    # seperate target...
    y = df['sold_quantity']

    # ...from exploratory data 
    X = df.drop('sold_quantity', axis=1)

    # create train test splits
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False, random_state=41)

    print(f'{name}:')
    print()

    # model

    # lightgbm
    lgb = lightgbm.LGBMRegressor(n_estimators=1000, subsample_for_bin=100000, random_state=42, metric='rmse')
    lgb.fit(X_train, y_train)
    
    y_pred = lgb.predict(X_test)

    # base errors for benchmark
    base_mae = mean_absolute_error(y_test, y_pred) if name == 'df_orig' else base_mae
    base_rmse = root_mean_squared_error(y_test, y_pred) if name == 'df_orig' else base_rmse
    base_wmape = weighted_mean_absolute_percentage_error(y_test, y_pred) if name == 'df_orig' else base_wmape

    # current loop calculations
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    wmape = weighted_mean_absolute_percentage_error(y_test, y_pred)

    # differences for benchmarks
    diff_mae = base_mae - mae
    diff_rmse = base_rmse - rmse
    diff_wmape = base_wmape - wmape

    # print results
    print()
    print(f'>> RESULTS {name}:')
    print(f'MAE: {mae} ({diff_mae})')
    print(f'RMSE: {rmse} ({diff_rmse})')
    print(f'WMAPE: {wmape} ({diff_wmape})')
   
    print('-'*60)

    
    
    

df_orig:

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.914529 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5860
[LightGBM] [Info] Number of data points in the train set: 2587352, number of used features: 30
[LightGBM] [Info] Start training from score 18.286266

>> RESULTS df_orig:
MAE: 3.7405226798305975 (0.0)
RMSE: 12.651348337886967 (0.0)
WMAPE: 0.2168582310623716 (0.0)
------------------------------------------------------------
df_events:

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categori

# __Error Investigation__

## __Train_Test_Set__

In [87]:
# load
df = pd.read_parquet('../data/processed/02_df_cleaned.parquet', engine='pyarrow')

In [88]:
cols_to_change = df.select_dtypes('int').columns
df[cols_to_change] = df[cols_to_change].astype('category')
df.dtypes

store_id                 category
item_id                  category
date               datetime64[ns]
category_name            category
sold_quantity             float32
revenue                   float32
zipcode                  category
holiday_name             category
holiday_type             category
wind_dir                 category
weather_code             category
temperature               float32
wind_speed                float32
wind_degree               float32
precip                    float32
humidity                  float32
visibility                float32
pressure                  float32
cloudcover                float32
lag_1                     float32
lag_7                     float32
lag_14                    float32
lag_28                    float32
rolling_mean_7            float32
rolling_std_7             float32
rolling_mean_14           float32
rolling_std_14            float32
rolling_mean_28           float32
rolling_std_28            float32
rolling_sum_7 

In [89]:
# create train and test data


# seperate target...
y = df['sold_quantity']

# ...from exploratory data 
X = df.drop(columns=['sold_quantity', 'revenue'])

# create train test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False, random_state=41)

X_date_test = X_test.date
X_date_train = X_train.date

X_train = X_train.drop('date', axis=1)
X_test = X_test.drop('date', axis=1)


## __Modeling__

In [90]:
# model

# lightgbm
lgb = lightgbm.LGBMRegressor(n_estimators=1000, subsample_for_bin=100000, random_state=42, metric='rmse')

lgb.fit(X_train, y_train)

y_pred = lgb.predict(X_test)

print('MSE:', mean_absolute_error(y_test, y_pred))
print('RMSE:', root_mean_squared_error(y_test, y_pred))
print('WMAPE:', weighted_mean_absolute_percentage_error(y_test, y_pred))

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.296926 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5860
[LightGBM] [Info] Number of data points in the train set: 2587352, number of used features: 30
[LightGBM] [Info] Start training from score 18.286266
MSE: 3.740522679830596
RMSE: 12.651348337886967
WMAPE: 0.2168582310623715


## __Error Investigation on category_name__

In [91]:
# creating a evaluation dataframe
df_eval = X_test[['store_id', 'item_id', 'category_name']]

dict_eval = {
    'date': X_date_test,
    'y_test': y_test,
    'y_pred': y_pred
}

df_eval_2 = pd.DataFrame(dict_eval)

# error datafame
df_error = pd.concat([df_eval, df_eval_2 ], axis=1).reset_index(drop=True)

In [92]:
# calculate error between true and pred
df_error['errors'] = df_error.y_test - df_error.y_pred

# errors by vategory_name
df_wmape = df_error.groupby('category_name').apply(lambda x: weighted_mean_absolute_percentage_error(x.y_test, x.y_pred))

# means 
dfe = df_error.groupby('category_name').agg(y_test_median=('y_test', 'median'), y_pred_median=('y_pred', 'median'), error_mean=('errors', 'mean'))
pd.concat([dfe, df_wmape], axis=1).rename(columns={0: 'wmape'}).sort_values('wmape')

,y_test_median,y_pred_median,error_mean,wmape
category_name,,,,
Brötchen,20.000,19.800945,0.781062,0.180465
Snack,6.000,6.485923,0.071665,0.232973
Brot,4.000,3.972679,0.087263,0.278143
Heißgetränke,4.000,4.309729,0.091607,0.282908
Feinbäckerei,7.000,6.888841,0.251202,0.285535
Angebot Feinbäckerei,6.000,5.918975,0.116759,0.348249
Angebot Brötchen,3.000,5.019204,0.449012,0.351955
Kuchen,2.500,2.722924,0.084571,0.396590
Konditorei,5.000,4.984963,0.204680,0.416907
